# Issue #2 — Collect Data and Construct Treasury Futures Roll Pairs

Download ZN daily OHLCV and statistics (open interest) from Databento, identify quarterly roll pairs, and compute First Notice Days.


In [14]:
from getpass import getpass
DATABENTO_API_KEY = getpass('')
import databento as db
import pandas as pd
import numpy as np
import re
client = db.Historical(DATABENTO_API_KEY)

## 1. Download daily OHLCV and open-interest data


In [15]:
# Daily OHLCV for all ZN outrights + calendar spreads (2020–2026)
ohlcv_data = client.timeseries.get_range(
    dataset='GLBX.MDP3', symbols=['ZN.FUT'], stype_in='parent',
    schema='ohlcv-1d', start='2020-01-01', end='2026-10-01',
)
ohlcv = ohlcv_data.to_df().reset_index()
# Index may be ts_event or ts_recv depending on schema — normalize
date_col = 'ts_event' if 'ts_event' in ohlcv.columns else ohlcv.columns[0]
ohlcv['date'] = pd.to_datetime(ohlcv[date_col]).dt.date
print(f'OHLCV: {len(ohlcv):,} rows, symbols: {ohlcv["symbol"].nunique()}')

# Statistics (open interest = stat_type 9)
stats_data = client.timeseries.get_range(
    dataset='GLBX.MDP3', symbols=['ZN.FUT'], stype_in='parent',
    schema='statistics', start='2020-01-01', end='2026-10-01',
)
stats = stats_data.to_df().reset_index()
date_col_s = 'ts_event' if 'ts_event' in stats.columns else stats.columns[0]
print(f'Statistics: {len(stats):,} rows')


C:\Users\84324\AppData\Local\Temp\ipykernel_14924\4243966385.py:2: BentoWarning: The streaming request contained one or more days which have reduced quality: 2020-02-27 (degraded), 2020-02-28 (degraded), 2020-05-05 (degraded)... See: https://databento.com/docs/api-reference-historical/metadata/metadata-get-dataset-condition
  ohlcv_data = client.timeseries.get_range(


OHLCV: 7,916 rows, symbols: 837


C:\Users\84324\AppData\Local\Temp\ipykernel_14924\4243966385.py:13: BentoWarning: The streaming request contained one or more days which have reduced quality: 2020-02-27 (degraded), 2020-02-28 (degraded), 2020-05-05 (degraded)... See: https://databento.com/docs/api-reference-historical/metadata/metadata-get-dataset-condition
  stats_data = client.timeseries.get_range(


Statistics: 506,462 rows


## 2. Extract open interest per contract per day


In [16]:
# Filter to open interest (stat_type = 9), keep last update per symbol per day
oi_raw = stats[stats['stat_type'] == 9].copy()
oi_raw['date'] = pd.to_datetime(oi_raw[date_col_s]).dt.date
sort_col = 'ts_recv' if 'ts_recv' in oi_raw.columns else date_col_s
oi = oi_raw.sort_values(sort_col).groupby(['symbol', 'date']).last()['quantity'].reset_index()
oi.columns = ['symbol', 'date', 'open_interest']
oi = oi[oi['open_interest'] < 1e15]
print(f'OI records: {len(oi):,}')
oi.head()


OI records: 5,334


,symbol,date,open_interest
0,ZNH0,2020-01-01,3604076
1,ZNH0,2020-01-02,3604076
2,ZNH0,2020-01-03,3644002
3,ZNH0,2020-01-05,3665107
4,ZNH0,2020-01-06,3664618


## 3. Classify symbols into outrights vs. spreads


In [17]:
# Outright pattern: ZN + month_code + year_digit (e.g. ZNH4, ZNM5)
# Spread pattern: contains '-' (e.g. ZNH4-ZNM4)
all_symbols = ohlcv['symbol'].unique()

outright_pat = re.compile(r'^ZN[HMUZ]\d$')
outrights = sorted([s for s in all_symbols if outright_pat.match(s)])
spreads   = sorted([s for s in all_symbols if '-' in s and s.startswith('ZN')])

print(f'Outrights ({len(outrights)}): {outrights[:10]} ...')
print(f'Spreads   ({len(spreads)}): {spreads[:10]} ...')


Outrights (29): ['ZNH0', 'ZNH1', 'ZNH2', 'ZNH3', 'ZNH4', 'ZNH5', 'ZNH6', 'ZNH7', 'ZNM0', 'ZNM1'] ...
Spreads   (55): ['ZNH0-ZNM0', 'ZNH1-N1UH1', 'ZNH1-ZNM1', 'ZNH2-N1UH2', 'ZNH2-ZNM2', 'ZNH3-ZNM3', 'ZNH3-ZNU3', 'ZNH4-ZNM4', 'ZNH4-ZNU4', 'ZNH5-ZNM5'] ...


## 4. Build quarterly roll pairs and First Notice Days

ZN delivers in March (H), June (M), September (U), December (Z). First Notice Day = last business day of the month before the contract month.


In [18]:
MONTH_CODES = {'H': 3, 'M': 6, 'U': 9, 'Z': 12}
NEXT_CODE   = {'H': 'M', 'M': 'U', 'U': 'Z', 'Z': 'H'}

def contract_to_delivery(sym):
    """Parse 'ZNH4' → (month_code, delivery_year, delivery_month)."""
    code, yr_digit = sym[2], int(sym[3])
    decade = 2020 if yr_digit >= 0 else 2020  # works for 2020-2029
    year = decade + yr_digit
    return code, year, MONTH_CODES[code]

def first_notice_day(year, delivery_month):
    """Last business day of the month before delivery_month."""
    prev_month = delivery_month - 1 if delivery_month > 1 else 12
    prev_year  = year if delivery_month > 1 else year - 1
    # Last day of prev_month
    if prev_month == 12:
        last_day = pd.Timestamp(prev_year, 12, 31)
    else:
        last_day = pd.Timestamp(prev_year, prev_month + 1, 1) - pd.Timedelta(days=1)
    # Roll back to business day
    fnd = pd.Timestamp(last_day)
    while fnd.weekday() >= 5:
        fnd -= pd.Timedelta(days=1)
    return fnd.date()

# Build roll pairs: front → next for each quarterly contract
rolls = []
for sym in outrights:
    mc, yr, dm = contract_to_delivery(sym)
    nc = NEXT_CODE[mc]
    next_yr = yr if nc != 'H' else yr + 1
    next_sym = f'ZN{nc}{next_yr % 10}'
    fnd = first_notice_day(yr, dm)
    if next_sym in outrights:
        rolls.append({'front': sym, 'next': next_sym, 'month_code': mc,
                      'delivery_year': yr, 'delivery_month': dm,
                      'fnd': fnd})

rolls_df = pd.DataFrame(rolls).sort_values(['delivery_year', 'delivery_month']).reset_index(drop=True)
print(f'{len(rolls_df)} roll pairs constructed')
rolls_df


28 roll pairs constructed


,front,next,month_code,delivery_year,delivery_month,fnd
0,ZNH0,ZNM0,H,2020,3,2020-02-28
1,ZNM0,ZNU0,M,2020,6,2020-05-29
2,ZNU0,ZNZ0,U,2020,9,2020-08-31
3,ZNZ0,ZNH1,Z,2020,12,2020-11-30
4,ZNH1,ZNM1,H,2021,3,2021-02-26
5,ZNM1,ZNU1,M,2021,6,2021-05-31
6,ZNU1,ZNZ1,U,2021,9,2021-08-31
7,ZNZ1,ZNH2,Z,2021,12,2021-11-30
8,ZNH2,ZNM2,H,2022,3,2022-02-28
9,ZNM2,ZNU2,M,2022,6,2022-05-31


## 5. Save raw data


In [19]:
os.makedirs('../data/raw', exist_ok=True)

# Save OHLCV (outrights only + spreads)
ohlcv_out = ohlcv[['date', 'symbol', 'open', 'high', 'low', 'close', 'volume']].copy()
ohlcv_out.to_parquet('../data/raw/zn_ohlcv_daily.parquet', index=False)

# Save open interest
oi.to_parquet('../data/raw/zn_open_interest.parquet', index=False)

# Save roll pairs
rolls_df.to_parquet('../data/raw/zn_roll_pairs.parquet', index=False)

print('Saved to data/raw/:')
for f in os.listdir('../data/raw'):
    size = os.path.getsize(f'../data/raw/{f}') / 1e6
    print(f'  {f}: {size:.1f} MB')


Saved to data/raw/:
  zn_ohlcv_daily.parquet: 0.1 MB
  zn_open_interest.parquet: 0.0 MB
  zn_roll_pairs.parquet: 0.0 MB
